# Data Pull

This notebook downloads Premier League data and saves it as csv files, so the other notebooks don't need to re-download anything.

**Data Sources:**
- **Understat**: has its own expected goals (xG) model. This project will make use of team xG, player xG and shot locations.
- **Fbref**: a stats site covering a wide range of metrics such as passing, defensive actions, minutes played, and more. Used here for as a secondary source.

**IMPORTANT**: Only run this notebook **once**. Everything will be saved on your computer locally afterwards, so there's no need to re-run it over and over again. The first run may also take a few minutes.

In [2]:
import soccerdata as sd
import pandas as pd
from pathlib import Path

[09/23/26 22:35:43] INFO     No custom team name replacements found. You can configure these in       ]8;id=12701224;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=12701225;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\damda\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=12701231;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=12701232;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\soccerdata\_config.py#189\189]8;;\
                             C:\Users\damda\soccerdata\config\league_dict.json.                                    

In [3]:
TEAM = "Sunderland"
LEAGUE = "ENG-Premier League"
SEASON = "2025-2026"
DATA_FOLDER = Path("data")
DATA_FOLDER.mkdir(parents=True, exist_ok=True) # create folder if it doesn't already exist

understat = sd.Understat(leagues=LEAGUE, seasons=SEASON)

# one row per match - home_team, away_team, home_xg, away_xg, home_goals, away_goals, and the date
understat_team_stats = understat.read_team_match_stats().reset_index()
understat_player_stats = understat.read_player_season_stats().reset_index()

# save tables to csv files
understat_team_stats.to_csv(DATA_FOLDER / "understat_team_stats.csv", index=False)
understat_player_stats.to_csv(DATA_FOLDER / "understat_player_stats.csv", index=False)


print("Team match stats found: ", understat_team_stats.shape)
print("Player season stats found: ", understat_player_stats.shape)

[09/23/26 22:35:46] INFO     Saving cached data to C:\Users\damda\soccerdata\data\Understat          ]8;id=12701239;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12701240;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-09-23 22:35:47] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


[09/23/26 22:35:47] INFO     Successfully loaded TLS library:                                      ]8;id=12701247;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=12701248;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packa                 
                             ges\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll                          

Team match stats found:  (380, 29)
Player season stats found:  (537, 23)


## Shot events

One row per shot, with pitch location, xG value and result (goal, saved, missed). This is the slowest download since it fetches match by match.

In [4]:
# shot event - one row per shot with location on pitch, xg value, and result (goal, saved, missed)

# slow download as it fetches match by match

try:
    understat_shots = understat.read_shot_events().reset_index()
    understat_shots.to_csv(DATA_FOLDER / "understat_shots.csv", index=False)
    print("Shots :", understat_shots.shape)
except Exception as shot_download_error:
    understat_shots = None
    print("Shot events download failed: ", shot_download_error)

Shots : (9524, 21)


## Fbref: general player stats

Fbref splits data into separate tables by stat type. We pull two that are called "standard" and "misc".
- standard: minutes played, position, goals, assists, progressive passes
- misc: cards, fouls, aerial duels, interceptions

These stats get merged into one table, for one row per player.

**Why a browser pops up**: Fbref uses bot protection so it blocks plain scripted requests such as "checking your browser" page instead of real data. To get around this soccerdata open an actual temporary Chrome browser like a human would. This is slow and fragile so closing the window or chrome updating can break the connection.

In [5]:
# fbref srcaper
fbref = sd.FBref(leagues=LEAGUE, seasons=SEASON)

#  four columns together identify one player in one team in one season
merge_key_columns = ["league", "season", "team", "player"]

# the two FBref tables we want
fbref_stat_types = ["standard", "misc"]

# dict to hold each downloaded table, labelled by its stat type.
fbref_tables_by_stat_type = {}

for stat_type in fbref_stat_types:
    try:
        # turn the row labels into normal columns.
        downloaded_table = fbref.read_player_season_stats(stat_type=stat_type)
        downloaded_table = downloaded_table.reset_index()

        flattened_column_names = []

        for column_header_pair in downloaded_table.columns:

            useful_parts = []
            for header_part in column_header_pair:
                header_text = str(header_part)
                is_empty = (header_text == "")
                is_placeholder = header_text.startswith("Unnamed")
                if not is_empty and not is_placeholder:
                    useful_parts.append(header_text)
            flattened_column_names.append("_".join(useful_parts))

        downloaded_table.columns = flattened_column_names

        fbref_tables_by_stat_type[stat_type] = downloaded_table
        print(f"FBref '{stat_type}' downloaded, (rows, columns):", downloaded_table.shape)

    except Exception as fbref_download_error:
        print(f"FBref '{stat_type}' failed:", fbref_download_error)

fbref_players_merged = None

if len(fbref_tables_by_stat_type) > 0:

    for stat_type, table in fbref_tables_by_stat_type.items():

        if fbref_players_merged is None:
            fbref_players_merged = table
        else:
            fbref_players_merged = fbref_players_merged.merge(
                table,
                on=merge_key_columns,
                how="left",
                suffixes=("", "_dup"), # so existing player doesn't get overwritten
            )

    # remove those  duplicate columns created above - the ones ending in "_dup"
    columns_to_keep = [
        column_name
        for column_name in fbref_players_merged.columns
        if not column_name.endswith("_dup")
    ]
    fbref_players_merged = fbref_players_merged[columns_to_keep]

    # save for the defenders
    fbref_players_merged.to_csv(DATA_FOLDER / "fbref_players.csv", index=False)
    print("FBref merged table (rows, columns):", fbref_players_merged.shape)

else:
    print("No FBref tables downloaded, so the defenders notebook will not work yet.")

[09/23/26 22:35:57] INFO     Saving cached data to C:\Users\damda\soccerdata\data\FBref              ]8;id=12701253;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12701254;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[09/23/26 22:36:05] WARNING  C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-pa ]8;id=12701261;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=12701262;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\_py_warnings.py#230\230]8;;\
                             ckages\soccerdata\fbref.py:171: FutureWarning: The behavior of                        
                             DataFrame concatenation with empty or all-NA entries is                               
                             deprecated. In a future version, this will no longer exclude empty                    
                             or all-NA columns when determining the result dtypes. To retain                       
                             the old behavior, exclude the relevant entries before the concat                      
                             operation.                                                                            
                               pd.concat(dfs)                                                                      
                                                                                                                   

FBref 'standard' downloaded, (rows, columns): (551, 25)


                    WARNING  C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\site-pa ]8;id=12701267;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=12701268;file://C:\Users\damda\AppData\Local\Programs\Python\Python314\Lib\_py_warnings.py#230\230]8;;\
                             ckages\soccerdata\fbref.py:171: FutureWarning: The behavior of                        
                             DataFrame concatenation with empty or all-NA entries is                               
                             deprecated. In a future version, this will no longer exclude empty                    
                             or all-NA columns when determining the result dtypes. To retain                       
                             the old behavior, exclude the relevant entries before the concat                      
                             operation.                                                                            
                               pd.concat(dfs)                                                                      
                                                                                                                   

FBref 'misc' downloaded, (rows, columns): (551, 21)
FBref merged table (rows, columns): (551, 36)


##  Merge datasets

And print every dataset's shape and column name so you know exactly what is available.

In [8]:
# merge all four datasets into a dict
all_datasets = {
    "understat_team_matches": understat_team_stats,
    "understat_players": understat_player_stats,
    "understat_shots": understat_shots,
    "fbref_players": fbref_players_merged,
}

for dataset_name, dataset_table in all_datasets.items():

    print("=" * 60)   # a line of "=" signs to separate the datasets visually

    if dataset_table is None:
        print(dataset_name, ": NOT AVAILABLE (download failed)")
        continue

    print(dataset_name, "-> (rows, columns):", dataset_table.shape)
    print("Columns:", list(dataset_table.columns))

understat_team_matches -> (rows, columns): (380, 29)
Columns: ['league', 'season', 'game', 'league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code', 'home_team_code', 'home_points', 'home_expected_points', 'home_goals', 'home_xg', 'home_np_xg', 'home_np_xg_difference', 'home_ppda', 'home_deep_completions', 'away_points', 'away_expected_points', 'away_goals', 'away_xg', 'away_np_xg', 'away_np_xg_difference', 'away_ppda', 'away_deep_completions']
understat_players -> (rows, columns): (537, 23)
Columns: ['league', 'season', 'team', 'player', 'league_id', 'season_id', 'team_id', 'player_id', 'position', 'matches', 'minutes', 'goals', 'xg', 'np_goals', 'np_xg', 'assists', 'xa', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'xg_chain', 'xg_buildup']
understat_shots -> (rows, columns): (9524, 21)
Columns: ['league', 'season', 'game', 'team', 'player', 'league_id', 'season_id', 'game_id', 'date', 'shot_id', 'team_id', 'player